Load Config

In [1]:
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd().parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "images.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

image_config = config["images"]

INPUT_DIR = PROJECT_ROOT / image_config["input_directory"]
OUTPUT_DIR = PROJECT_ROOT / image_config["output_directory"]

WIDTH = image_config["output_size"]["width"]
HEIGHT = image_config["output_size"]["height"]

print("Input directory:", INPUT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Output size:", WIDTH, "x", HEIGHT)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Input directory: c:\Users\vanle\Projects\aolmlops\data\raw\images
Output directory: c:\Users\vanle\Projects\aolmlops\data\processed\images
Output size: 224 x 224


Implement Resizing

In [2]:
from PIL import Image, ImageOps

def preprocess_image(image_path, output_path, config):
    width = config["output_size"]["width"]
    height = config["output_size"]["height"]

    with Image.open(image_path) as img:

        img = img.convert(config["color_mode"])

        resized = ImageOps.contain(
            img,
            (width, height),
            method=Image.Resampling.LANCZOS
        )

        padding_color = tuple(config["padding_color"])

        canvas = Image.new(
            "RGB",
            (width, height),
            padding_color
        )

        x = (width - resized.width) // 2
        y = (height - resized.height) // 2

        canvas.paste(resized, (x, y))

        canvas.save(output_path, format=config["format"])

Process Images

In [3]:
from tqdm.auto import tqdm

image_files = sorted(INPUT_DIR.rglob("*.jpg"))

processed_count = 0
processing_errors = []

for image_path in tqdm(image_files, desc="Processing images"):

    # Preserve the original 10-digit filename stem
    article_id = image_path.stem

    output_filename = image_config["filename_format"].format(
        article_id=article_id
    )

    output_path = OUTPUT_DIR / output_filename

    try:
        preprocess_image(
            image_path,
            output_path,
            image_config
        )

        processed_count += 1

    except Exception as e:
        processing_errors.append({
            "article_id": article_id,
            "image_path": str(image_path),
            "error": str(e)
        })

print("Successfully processed:", processed_count)
print("Processing errors:", len(processing_errors))

Processing images:   0%|          | 0/105100 [00:00<?, ?it/s]

Successfully processed: 105100
Processing errors: 0


Save Preprocess Report

In [4]:
import polars as pl

processing_error_report = pl.DataFrame(
    processing_errors,
    schema={
        "article_id": pl.String,
        "image_path": pl.String,
        "error": pl.String
    }
)

processing_error_report.write_parquet(
    PROJECT_ROOT / "data" / "processed" / "image_processing_errors.parquet"
)

print("Saved image processing error report.")

Saved image processing error report.


Validation, Check Dimensions, Format, Color Mode

In [5]:
from collections import Counter

processed_files = sorted(OUTPUT_DIR.glob("*.jpg"))

processed_dimensions = Counter()
processed_modes = Counter()
processed_formats = Counter()
validation_errors = []

for path in tqdm(processed_files, desc="Validating processed images"):
    try:
        with Image.open(path) as img:
            processed_dimensions[img.size] += 1
            processed_modes[img.mode] += 1
            processed_formats[img.format] += 1

            if img.size != (WIDTH, HEIGHT):
                validation_errors.append({
                    "image": path.name,
                    "error": f"Incorrect dimensions: {img.size}"
                })

            if img.mode != "RGB":
                validation_errors.append({
                    "image": path.name,
                    "error": f"Incorrect color mode: {img.mode}"
                })

            if img.format != "JPEG":
                validation_errors.append({
                    "image": path.name,
                    "error": f"Incorrect format: {img.format}"
                })

            img.verify()

    except Exception as e:
        validation_errors.append({
            "image": path.name,
            "error": str(e)
        })

print("Total processed images:", len(processed_files))
print("Dimensions:", processed_dimensions)
print("Color modes:", processed_modes)
print("Formats:", processed_formats)
print("Validation errors:", len(validation_errors))

Validating processed images:   0%|          | 0/105100 [00:00<?, ?it/s]

Total processed images: 105100
Dimensions: Counter({(224, 224): 105100})
Color modes: Counter({'RGB': 105100})
Formats: Counter({'JPEG': 105100})
Validation errors: 0


Save Validation Report

In [6]:
validation_error_report = pl.DataFrame(
    validation_errors,
    schema={
        "image": pl.String,
        "error": pl.String
    }
)

validation_error_report.write_parquet(
    PROJECT_ROOT / "data" / "processed" / "image_validation_errors.parquet"
)

print("Saved processed-image validation report.")

Saved processed-image validation report.


Mapping for Processed Images

In [7]:
#Load original article image mapping from EDA 
import polars as pl
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

article_image_mapping = pl.read_parquet(
    PROJECT_ROOT / "data" / "interim" / "article_image_mapping.parquet"
)

print("Loaded article-image mapping.")
print("Rows:", article_image_mapping.height)
print(article_image_mapping.head())

Loaded article-image mapping.
Rows: 105542
shape: (5, 2)
┌────────────┬─────────────────────────────────┐
│ article_id ┆ image_path                      │
│ ---        ┆ ---                             │
│ i64        ┆ str                             │
╞════════════╪═════════════════════════════════╡
│ 108775015  ┆ ..\data\raw\images\010\0108775… │
│ 108775044  ┆ ..\data\raw\images\010\0108775… │
│ 108775051  ┆ ..\data\raw\images\010\0108775… │
│ 110065001  ┆ ..\data\raw\images\011\0110065… │
│ 110065002  ┆ ..\data\raw\images\011\0110065… │
└────────────┴─────────────────────────────────┘


In [8]:
processed_image_index = pl.DataFrame({
    "image_filename": [path.name for path in processed_files],
    "processed_image_path": [
        f"data/processed/images/{path.name}"
        for path in processed_files
    ]
}).with_columns(
    pl.col("image_filename")
    .str.replace(".jpg", "")
    .cast(pl.Int64)
    .alias("article_id")
)

processed_image_mapping = article_image_mapping.join(
    processed_image_index.select([
        "article_id",
        "processed_image_path"
    ]),
    on="article_id",
    how="left"
)

processed_image_mapping.write_parquet(
    PROJECT_ROOT / "data" / "processed" / "article_processed_image_mapping.parquet"
)

print("Saved processed-image mapping.")

Saved processed-image mapping.


In [9]:
print("Total articles:", processed_image_mapping.height)

print(
    "Articles with processed images:",
    processed_image_mapping["processed_image_path"].is_not_null().sum()
)

print(
    "Articles without processed images:",
    processed_image_mapping["processed_image_path"].is_null().sum()
)

print(
    "Unique article IDs:",
    processed_image_mapping["article_id"].n_unique()
)

Total articles: 105542
Articles with processed images: 105100
Articles without processed images: 442
Unique article IDs: 105542


In [10]:
#Check for missing physical files in the processed image mapping
from pathlib import Path

processed_paths = [
    PROJECT_ROOT / path
    for path in processed_image_mapping
        ["processed_image_path"]
        .drop_nulls()
        .to_list()
]

existing_images = sum(path.is_file() for path in processed_paths)
missing_files = [path for path in processed_paths if not path.is_file()]

print("Expected processed images:", len(processed_paths))
print("Existing processed images:", existing_images)
print("Missing physical files:", len(missing_files))

Expected processed images: 105100
Existing processed images: 105100
Missing physical files: 0


In [11]:
#Final summary of the image processing pipeline
image_pipeline_summary = pl.DataFrame({
    "metric": [
        "total_articles",
        "articles_with_original_images",
        "articles_without_original_images",
        "processed_images",
        "missing_processed_files",
        "output_width",
        "output_height",
    ],
    "value": [
        processed_image_mapping.height,
        article_image_mapping["image_path"].is_not_null().sum(),
        article_image_mapping["image_path"].is_null().sum(),
        len(processed_files),
        len(missing_files),
        WIDTH,
        HEIGHT,
    ]
})

image_pipeline_summary.write_parquet(
    PROJECT_ROOT / "data" / "processed" / "image_pipeline_summary.parquet"
)

print(image_pipeline_summary)

shape: (7, 2)
┌─────────────────────────────────┬────────┐
│ metric                          ┆ value  │
│ ---                             ┆ ---    │
│ str                             ┆ i64    │
╞═════════════════════════════════╪════════╡
│ total_articles                  ┆ 105542 │
│ articles_with_original_images   ┆ 105100 │
│ articles_without_original_imag… ┆ 442    │
│ processed_images                ┆ 105100 │
│ missing_processed_files         ┆ 0      │
│ output_width                    ┆ 224    │
│ output_height                   ┆ 224    │
└─────────────────────────────────┴────────┘


In [12]:
#Final image pipeline verification
assert processed_image_mapping.height == 105542

assert (
    processed_image_mapping["processed_image_path"]
    .is_not_null()
    .sum()
    == 105100
)

assert (
    processed_image_mapping["processed_image_path"]
    .is_null()
    .sum()
    == 442
)

assert len(missing_files) == 0

assert len(processing_errors) == 0

assert len(validation_errors) == 0

print("IMAGE PIPELINE: ALL CHECKS PASSED")

IMAGE PIPELINE: ALL CHECKS PASSED
